<a href="https://colab.research.google.com/github/samarreguigui/Computerlinguistik/blob/main/Parser10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
def next_char(parse):
    i = 0
    while i < len(parse) and parse[i].isspace():
        i += 1
    if i >= len(parse):
        raise Exception("unexpected end", parse)
    return parse[i], parse[i:]


def consume_char(parse):
    ch, rest = next_char(parse)
    return rest[1:], ch


def read_label(parse):
    ch, rest = next_char(parse)
    if ch in "()":
        raise Exception("invalid label", parse)

    i = 0
    while i < len(rest) and not rest[i].isspace() and rest[i] not in "()":
        i += 1

    label = rest[:i]
    if not label:
        raise Exception("empty label", parse)

    return rest[i:], label


def read_tree(parse, word_list=None, const_list=None):
    if word_list is None:
        word_list = []
    if const_list is None:
        const_list = []

    parse, ch = consume_char(parse)
    if ch != "(":
        raise Exception("expected '('", parse)

    parse, label = read_label(parse)

    start = len(word_list)
    idx = len(const_list)
    const_list.append([label, start, None])

    ch, _ = next_char(parse)
    if ch == "(":
        while True:
            ch, _ = next_char(parse)
            if ch == ")":
                break
            parse, word_list, const_list = read_tree(parse, word_list, const_list)
        parse, _ = consume_char(parse)
    else:
        parse, word = read_label(parse)
        word_list.append(word)
        parse, _ = consume_char(parse)

    const_list[idx][2] = len(word_list)
    return parse, word_list, const_list


def build_tree(parse, word_list, const_list):
    label, start, end = const_list.pop(0)
    parse += "(" + label

    if end == start + 1:
        parse += " " + word_list[start] + ")"
        return parse, word_list, const_list

    while const_list and const_list[0][1] < end:
        parse, word_list, const_list = build_tree(parse, word_list, const_list)

    parse += ")"
    return parse, word_list, const_list


def parse(parse_string):
    try:
        rest, word_list, const_list = read_tree(parse_string)
        if rest.strip():
            raise Exception("trailing characters", rest)

        i = 0
        collapsed = []
        while i < len(const_list):
            label, s, e = const_list[i]
            j = i + 1
            while j < len(const_list) and const_list[j][1] == s and const_list[j][2] == e:
                label = label + "=" + const_list[j][0]
                j += 1
            collapsed.append((label, s, e))
            i = j
        const_list = collapsed

        tree, _, _ = build_tree("", word_list, const_list.copy())
        return parse_string, word_list, const_list, tree

    except Exception as e:
        message, rest = e.args
        pos = len(parse_string) - len(rest)
        print(parse_string)
        print(" " * pos + "^")
        print(message)
        return None

In [6]:
def main(filename):
    with open(filename, encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            print("ORIGINAL")
            print(line)

            result = parse(line)
            if result is None:
                print()
                continue

            original, words, consts, rebuilt = result
            print("WORDS")
            print(words)
            print("CONSTITUENTS")
            print(consts)
            print("REBUILT")
            print(rebuilt)
            print()


In [7]:
main("/content/test.txt")


Streaming output truncated to the last 5000 lines.
CONSTITUENTS
[('TOP=S', 0, 28), ('NP=NP', 0, 2), ('DT', 0, 1), ('NNS', 1, 2), ('PP', 2, 4), ('IN', 2, 3), ('NP=NNP', 3, 4), ('VP', 4, 27), ('VBD', 4, 5), ('PP', 5, 12), ('IN', 5, 6), ('NP', 6, 12), ('NP', 6, 8), ('DT', 6, 7), ('NNS', 7, 8), ('PP', 8, 12), ('IN', 8, 9), ('NP', 9, 12), ('DT', 9, 10), ('NNP', 10, 11), ('NN', 11, 12), (':', 12, 13), ('NP', 13, 27), ('NP', 13, 18), ('ADJP', 13, 16), ('JJ', 13, 14), ('CC', 14, 15), ('JJ', 15, 16), ('JJ', 16, 17), ('NNS', 17, 18), (',', 18, 19), ('CONJP', 19, 22), ('RB', 19, 20), ('RB', 20, 21), ('IN', 21, 22), ('NP', 22, 27), ('NP', 22, 24), ('JJ', 22, 23), ('NNS', 23, 24), ('CC', 24, 25), ('NP', 25, 27), ('NN', 25, 26), ('NNS', 26, 27), ('.', 27, 28)]
REBUILT
(TOP=S(NP=NP(DT The)(NNS sellers))(PP(IN on)(NP=NNP Friday))(VP(VBD came)(PP(IN from)(NP(NP(DT all)(NNS corners))(PP(IN of)(NP(DT the)(NNP OTC)(NN market)))))(: --)(NP(NP(ADJP(JJ big)(CC and)(JJ small))(JJ institutional)(NNS investors)